## Exámen 1 - Práctico
### Objetivo general

La parte practica busca evaluar si el estudiante entiende el flujo completo de modelado visto en clase:

- revision inicial de datos
- analisis exploratorio
- analisis bivariado
- regresion lineal simple
- regresion lineal multiple
- regresion polinomial
- Ridge regression
- StandardScaler
- KNN
- cross validation
- comparacion de modelos usando R2

El objetivo no es solo obtener el mejor R2. El objetivo es que el estudiante explique que hizo, por que lo hizo, que encontro y que significa cada resultado.

## Regla obligatoria para ambos datasets

En cada seccion debe haber una conclusion escrita.

No basta con correr codigo, imprimir tablas o mostrar graficas. Despues de cada analisis o modelo, el estudiante debe explicar con sus palabras:

- que hizo
- que resultado obtuvo
- que significa ese resultado
- si el modelo o variable parece servir
- que decision tomaria con base en el resultado

Si una seccion no tiene interpretacion o conclusion, esta incompleta aunque el codigo funcione.

Dependiendo el que te toque (hitters o boston) 

la idea es que apliques todo lo que vimos en clase, hagas varioa modelos y des conclusiones



## 1. Revisión inicial de los datos

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import linear_model
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
df = pd.read_csv('../Data/Hitters_examen.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../Data/Hitters_examen.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()

In [ ]:
df.isna().sum()

### Conclusión de la revisión inicial

Primero revise el dataset y tenia 322 observaciones con 20 variables
vi que Salary tenia 59 valores faltantes y como es la variable que quiero predecir elimine esas filas
despues de eso quedaron 263 observaciones
tambien hay variables numericas y 3 categoricas que son League Division y NewLeague entonces mas adelante tengo que tomarlas en cuenta dependiendo del modelo

## 2. Análisis exploratorio

In [ ]:
df[['Salary']].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(df['Salary'], bins=18)
ax.set_title('Salary')

plt.show()

### Conclusión del análisis exploratorio

Revise Salary y vi que tiene un promedio de 535.93 y una mediana de 425
tambien tiene bastante variacion ya que va desde 67.5 hasta 2460
en el histograma se ve que hay mas jugadores con salarios bajos y pocos con salarios muy altos
entonces hay algunos valores altos que hacen que el promedio aumente
por eso no me basaria solamente en el promedio para entender Salary

## 3. Análisis bivariado

In [ ]:
df_bivariado = df.copy()

del df_bivariado['League']
del df_bivariado['Division']
del df_bivariado['NewLeague']

df_bivariado.corr()

In [ ]:
df_bivariado['CRBI_bin'] = pd.qcut(
    df_bivariado['CRBI'],
    q=10,
    labels=False
) + 1

bivariado_CRBI = df_bivariado.groupby('CRBI_bin').mean()

bivariado_CRBI[['CRBI', 'Salary']]

In [ ]:
plt.scatter(df.CRBI, df.Salary)
plt.show()

### Conclusión del análisis bivariado

En el analisis bivariado revise la relacion de las variables numericas con Salary
CRBI fue la variable con mayor correlacion con 0.566966 seguida de CRuns con 0.562678 y CHits con 0.548910
entonces CRBI parece ser la variable que mas informacion aporta por si sola y la voy a usar para comenzar con la regresion lineal simple

## 4. Regresión lineal simple

In [ ]:
X = df[['CRBI']]
y = df['Salary']

model = LinearRegression().fit(X,y)
predicciones = model.predict(X)
from sklearn.metrics import r2_score

r2_simple = r2_score(y_pred=predicciones, y_true=y)
r2_simple

In [ ]:
plt.scatter(df.CRBI, df.Salary)
plt.scatter(df.CRBI, predicciones)
plt.show()

### Conclusión de regresión lineal simple

Use CRBI para hacer la regresion lineal simple porque fue la variable que tuvo mayor correlacion con Salary
el modelo dio un R2 de 0.3215 entonces explica aproximadamente el 32.15% de la variacion de Salary
CRBI si aporta informacion pero todavia queda una parte bastante grande que el modelo no explica
por eso ahora probaria una regresion multiple usando mas variables

## 5. Regresión lineal múltiple

In [ ]:
df = pd.get_dummies(df)

In [ ]:
X = df.copy()
del X['Salary']
y = df['Salary']

model = linear_model.LinearRegression().fit(X,y)

predicciones_todo = model.predict(X)

r2_multiple = r2_score(y_pred=predicciones_todo, y_true=y)
r2_multiple

### Conclusión de regresión lineal múltiple

En la regresion lineal multiple use todas las variables para predecir Salary
el modelo dio un R2 de 0.5461 entonces explica aproximadamente el 54.61% de la variacion de Salary
este resultado fue mejor que la regresion simple que tenia un R2 de 0.3215
entonces agregar mas variables si ayudo bastante al modelo aunque todavia queda una parte de Salary que no logra explicar

## 6. Regresión polinomial

In [ ]:
X = df[['CRBI']]
y = df['Salary']

In [ ]:
# Regresion polinomial grado 2
degree = 2
poly_features = PolynomialFeatures(degree=degree)
X_poly = poly_features.fit_transform(X)

# Ajustar el modelo de regresión lineal
model = linear_model.LinearRegression()
model.fit(X_poly, y)

df['predicciones_polinimio2'] = model.predict(X_poly)

r2_poly = r2_score(
    y_pred=df.predicciones_polinimio2,
    y_true=y
)

r2_poly

In [ ]:
model = Pipeline([
    ('poly', PolynomialFeatures()),
    ('regression', linear_model.LinearRegression())
])

param_grid = {
    'poly__degree': range(1, 5)
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='r2'
)

grid.fit(X, y)

In [ ]:
grid.best_params_

### Conclusión de regresión polinomial

Probe una regresion polinomial usando CRBI para predecir Salary
el modelo de grado 2 dio un R2 de 0.4011 entonces explica aproximadamente el 40.11% de la variacion de Salary
este resultado fue mejor que la regresion lineal simple que tenia un R2 de 0.3215
despues use GridSearchCV para comparar los grados del 1 al 4 y el mejor fue grado 2
entonces permitir una relacion curva si mejoro el modelo
por eso vale la pena probar este modelo en la comparacion final

## 7. Ridge Regression

In [ ]:
X = df.copy()

del X['Salary']
del X['predicciones_polinimio2']

y = df['Salary']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
modelo_lineal = make_pipeline(StandardScaler(), LinearRegression())
modelo_ridge = make_pipeline(StandardScaler(), Ridge(alpha=30))

In [ ]:
modelo_lineal.fit(X_train, y_train)
modelo_ridge.fit(X_train, y_train)

pred_lineal = modelo_lineal.predict(X_test)
pred_ridge = modelo_ridge.predict(X_test)

In [ ]:
r2_lineal_test = r2_score(y_pred=pred_lineal, y_true=y_test)
r2_lineal_test

In [ ]:
r2_ridge = r2_score(y_pred=pred_ridge, y_true=y_test)
r2_ridge

### Conclusión de Ridge

Compare la regresion lineal con Ridge usando los mismos datos de entrenamiento y prueba
la regresion lineal obtuvo un R2 de 0.3806 y Ridge obtuvo 0.3631
entonces Ridge no mejoro el modelo porque su R2 fue menor
los resultados fueron parecidos pero la regresion lineal funciono mejor en este caso
por eso entre estos dos me quedaria con la regresion lineal

## 8. KNN

In [ ]:
modelo_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor())
])

In [ ]:
params = {
    'model__n_neighbors': range(1, 51)
}

In [ ]:
grid_knn = GridSearchCV(
    modelo_knn,
    params,
    cv=5,
    scoring='r2'
)

grid_knn.fit(X_train, y_train)

In [ ]:
grid_knn.best_params_

In [ ]:
pred_knn = grid_knn.predict(X_test)

r2_knn = r2_score(
    y_pred=pred_knn,
    y_true=y_test
)

r2_knn

### Conclusión de KNN

Para KNN use StandardScaler porque trabaja con distancias
despues use cross validation para probar diferentes valores de k y el mejor fue 13
con ese valor el modelo obtuvo un R2 de 0.3647 en los datos de prueba
entonces explica aproximadamente el 36.47% de la variacion de Salary
por eso voy a usar k igual a 13 para comparar KNN contra los demas modelos

## 9. Comparación de modelos usando R2

In [ ]:
#Regresion simple 
X_train_simple = X_train[['CRBI']]
X_test_simple = X_test[['CRBI']]

model_simple = LinearRegression().fit(X_train_simple, y_train)

pred_simple = model_simple.predict(X_test_simple)

r2_simple_test = r2_score(
    y_pred=pred_simple,
    y_true=y_test
)

r2_simple_test

In [ ]:
#Regresion multiple
model_multiple = linear_model.LinearRegression().fit(X_train, y_train)

pred_multiple = model_multiple.predict(X_test)

r2_multiple_test = r2_score(
    y_pred=pred_multiple,
    y_true=y_test
)

r2_multiple_test

In [ ]:
#Regresion polinomial
model_poly = Pipeline([
    ('poly', PolynomialFeatures()),
    ('regression', linear_model.LinearRegression())
])

In [ ]:
param_grid = {
    'poly__degree': range(1, 5)
}

In [ ]:
grid_poly = GridSearchCV(
    model_poly,
    param_grid,
    cv=5,
    scoring='r2'
)

grid_poly.fit(X_train_simple, y_train)

In [ ]:
grid_poly.best_params_

Para la comparacion final volvi a hacer cross validation usando solamente los datos de entrenamiento y en este caso el mejor grado fue 3

In [ ]:
pred_poly = grid_poly.predict(X_test_simple)

r2_poly_test = r2_score(
    y_pred=pred_poly,
    y_true=y_test
)

r2_poly_test

In [ ]:
#Ridge
r2_ridge

In [ ]:
#KNN
r2_knn

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': [
        'Regresion lineal simple',
        'Regresion lineal multiple',
        'Regresion polinomial grado 3',
        'Ridge',
        'KNN'
    ],
    'R2 prueba': [
        r2_simple_test,
        r2_multiple_test,
        r2_poly_test,
        r2_ridge,
        r2_knn
    ]
})

comparacion.sort_values('R2 prueba', ascending=False).round(3)

Al comparar todos los modelos usando los mismos datos de prueba la regresion lineal multiple fue la que obtuvo el mejor R2 con 0.381
despues quedo KNN con 0.365 y Ridge con 0.363
la regresion simple obtuvo 0.301 y la polinomial fue la mas baja con 0.166
entonces el modelo que mejor funciono para predecir Salary fue la regresion lineal multiple

## 10. Conclusión final

Primero revise el dataset y limpie los datos para poder trabajar solamente con observaciones completas
despues hice el analisis exploratorio de Salary y vi que habia bastante variacion entre los salarios y algunos valores muy altos
luego hice el analisis bivariado para revisar que variables tenian mayor relacion con Salary y CRBI fue la que tuvo la correlacion mas alta por eso la use para comenzar con la regresion lineal simple

La regresion lineal simple dio un R2 de 0.3215 entonces CRBI si aporta informacion pero por si sola no explica una parte tan grande de Salary
despues use todas las variables en una regresion lineal multiple y el resultado mejoro bastante lo que muestra que combinar varias variables ayuda mas que usar solamente CRBI

Tambien probe una regresion polinomial para ver si una relacion curva podia mejorar el modelo y use cross validation para elegir el grado
en la comparacion final el mejor grado fue 3 aunque su R2 en prueba fue de 0.166 entonces no generalizo bien con datos que no habia visto

Para Ridge use StandardScaler y lo compare con la regresion lineal usando los mismos datos de entrenamiento y prueba
Ridge obtuvo un R2 de 0.363 mientras que la regresion lineal obtuvo 0.381 entonces en este caso la regularizacion no mejoro el resultado

Despues probe KNN y tambien use StandardScaler porque este modelo depende de las distancias entre las observaciones
con cross validation el mejor valor fue k igual a 13 y obtuvo un R2 de 0.365 en prueba

Al final compare todos los modelos usando el mismo conjunto de prueba y la regresion lineal multiple fue la que obtuvo el mejor resultado con un R2 de 0.381 seguida de KNN con 0.365 y Ridge con 0.363
la regresion simple obtuvo 0.301 y la polinomial fue la mas baja con 0.166

Entonces el modelo que mejor funciono para predecir Salary fue la regresion lineal multiple
aun asi su R2 sigue siendo relativamente bajo entonces todavia hay una parte importante de Salary que estas variables no logran explicar
por eso me quedaria con la regresion lineal multiple de los modelos que probe aunque todavia se podria intentar mejorar el modelo con otras variables o diferentes formas de modelado